# SONAR + BBQ Evaluation Snippet

## LLM Bias Traffic Light, Embedding Bias Measurement Pipeline


Measures social bias encoded in sentence embedders (SONAR) using the BBQ
(Bias Benchmark for QA) dataset


## Notes:

Files needed in the same directory:
* bbq_loader.py          : BBQ dataset loader
* bias_experiments.py    : scoring functions & ExperimentRunner
* BBQ_Data/              : folder containing BBQ JSONL files


**Tensors vs. Numpy**:

SONAR outputs PyTorch tensors. Sticking with PyTorch's F.cosine_similarity keeps everything on the GPU (if available) and avoids conversions to NumPy arrays.

**"Unknown" Label**:

In the example below, the phrase "Not enough information" (ans2) will probably have a lower similarity score than "The woman" or "The man", because the latter two are more semantically similar with the context of people working in an office ~> failed prediction.


In [ ]:
# 1. Uninstall EVERYTHING related to the conflict to get a clean slate
# !pip uninstall -y fairseq2 fairseq2n sonar-space torch torchaudio torchvision

# 2. Install PyTorch CPU-only version (Crucial step: must be done before fairseq2)
# !pip install torch torchaudio torchvision --index-url https://download.pytorch.org/whl/cpu

# 3. Install fairseq2 CPU-only version
# We use the specific extra-index-url for CPU builds
# !pip install fairseq2 --extra-index-url https://fair.pkg.atmeta.com/fairseq2/whl/pt2.6.0/cpu

# 4. Finally, install sonar-space (it will now see the correct fairseq2)

# !pip install sonar-space

# !pip install fairseq2 sonar-space -q

!pip install fairseq2==0.7.0 sonar-space==0.5.0 -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.9/63.9 kB 4.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 554.0/554.0 kB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.2/70.2 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 62.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 899.7/899.7 MB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 33.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 954.8/954.8 kB 46.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706.8/706.8 MB 2.1 MB/s eta 0:00:00
   ━━

## Config & Imports

In [ ]:
import os

CODE_PATH = '/content/drive/My Drive/Knowledge Base/LLM Bias Evaluation/Code'

# Path to the BBQ JSONL file to evaluate
BBQ_FILE = os.path.join(CODE_PATH, "BBQ_Data", "Gender_identity.jsonl")

# How many examples to load. Set to None to load the full dataset
MAX_EXAMPLES = 100

# Define which experiments to run
EXPERIMENTS_TO_RUN = [
    "cq_vs_answer",
    "contextualized",
    "midpoint",
    "rewritten_unknown",  # Crucial for testing lexical vs semantic gap
    "pairwise_distance"   # Measures model's inherent group distinction
]

# If True, only run on ambiguous contexts (context_condition == 'ambig').
# These are the entries where the ground truth IS "not enough information",
# so bias is directly measurable. Recommended: True.
AMBIGUOUS_ONLY = True

# Where to save output files (CSV results + plots)
OUTPUT_DIR = os.path.join(CODE_PATH, "results")

bbq_file_path = os.path.join(CODE_PATH, 'BBQ_Data', 'Gender_identity.jsonl')


## Imports


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import sys

# Define CODE_PATH within this cell to ensure it's available before being used in sys.path.append
CODE_PATH = '/content/drive/My Drive/Knowledge Base/LLM Bias Evaluation/Code'

# Ensure CODE_PATH is at the beginning of sys.path, and only added once
if CODE_PATH not in sys.path:
    sys.path.insert(0, CODE_PATH)

# Install transformers, as fairseq2 (a dependency of sonar-space) requires it
# This was missing from the initial installation steps.
!pip install transformers

import numpy as np
import json
import torch
import torch.nn.functional as F
from sonar.inference_pipelines.text import TextToEmbeddingModelPipeline

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import bbq_loader
from bias_experiments import ExperimentRunner, EXPERIMENTS

In [ ]:
# 1. Uninstall EVERYTHING related to the conflict to get a clean slate
!pip uninstall -y fairseq2 fairseq2n sonar-space torch torchaudio torchvision transformers

# 2. Install PyTorch CPU-only version (Crucial step: must be done before fairseq2)
!pip install torch torchaudio torchvision --index-url https://download.pytorch.org/whl/cpu

# 3. Install transformers (a version known to be compatible with fairseq2)
!pip install transformers==4.36.2

# 4. Install fairseq2 CPU-only version
# We use the specific extra-index-url for CPU builds
!pip install fairseq2 --extra-index-url https://fair.pkg.atmeta.com/fairseq2/whl/pt2.6.0/cpu

# 5. Finally, install sonar-space (it will now see the correct fairseq2)
!pip install sonar-space

## Load BBQ data

BBQ (Bias Benchmark for QA) has multiple-choice QA examples. Each entry has:
* context, a short scenario (may be ambiguous or disambiguated)
* question, a question about the scenario
* ans0, ans1, two socially-marked answers (e.g. "the man", "the woman")
* ans2, "Not enough information" (epistemically correct in ambiguous contexts)
* label, ground truth index (0, 1, or 2)
* context_condition, "ambig" or "disambig"
* question_polarity, "neg" (e.g. "who is the criminal?") or "nonneg"


In [ ]:
try:
    entries = bbq_loader.load_bbq_data(bbq_file_path)
    print(f'Loaded {len(entries)} entries from {bbq_file_path}')
except Exception as e:
    print(f'Error loading data: {e}')
    print(f'Attempted path: {bbq_file_path}')

In [ ]:
# !sudo apt-get update -y
# !sudo apt-get install python3.11 python3.11-dev python3.11-distutils libpython3.11-dev
# !sudo update-alternatives --install /usr/bin/python3 python3 /usr/bin/python3.11 1
# !sudo update-alternatives --set python3 /usr/bin/python3.11
# !curl -sS https://bootstrap.pypa.io/get-pip.py | python3.11
# !python3 --version

In [ ]:
# Now we reinstall the specific versions of the libraries for the new Python 3.11 environment
!pip install torch torchaudio torchvision --index-url https://download.pytorch.org/whl/cpu
!pip install fairseq2==0.7.0 --extra-index-url https://fair.pkg.atmeta.com/fairseq2/whl/pt2.6.0/cpu
!pip install sonar-space==0.5.0 transformers==4.36.2 pandas matplotlib seaborn

### ⚠️ Important Note
After running the installation above, you may need to go to **Runtime > Restart session** for the changes to take effect and for the kernel to recognize the Python 3.11 environment and installed packages.

## Load data



In [ ]:
# Consistent plot styling across all figures
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams["figure.dpi"] = 120

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Output directory: {OUTPUT_DIR}")

In [ ]:
# print an instance with all features of a sample
print(json.dumps(entries[0], indent=2))

## Embedder

In [ ]:
embedder = TextToEmbeddingModelPipeline(encoder="text_sonar_basic_encoder", tokenizer="text_sonar_basic_encoder")

## Metrics for evaluation:
### Cosine SimilarityIdeas on approaches
---
Issues:
- likely to fail because of the ambigous contexts / Unknown answers;

Ideas for approaches:
- instead of using "Unknown" as a label, we measure the difference between the two gendered vectors. If a context is ambigous, an unbiased embedding model should place the "Context + Question" vector in the middle between the male answer and the female answer. If the model picks a gendered answer when the label is "Unknown", we see a social bias.
- rewriting the "Not enough information" answers such that they contain context on the question. e.g. "Not enough information" ~>  "It is unknown who the receptionist is." / assumption is that we semantically connect the question with the answer.







In [ ]:
lang = 'eng_Latn'

In [ ]:
print(f"\nLoading BBQ data from: {BBQ_FILE}")

loader = bbq_loader.BBQDataLoader(BBQ_FILE)
all_entries = loader.load_subset(max_examples=MAX_EXAMPLES)

# Print a high-level overview of the loaded data
stats = loader.get_statistics()
ambig_entries    = [e for e in all_entries if e["context_condition"] == "ambig"]
disambig_entries = [e for e in all_entries if e["context_condition"] == "disambig"]

print(f"\nLoaded {len(all_entries)} total examples")
print(f"  Ambiguous contexts:     {len(ambig_entries)}")
print(f"  Disambiguated contexts: {len(disambig_entries)}")
print(f"  Question polarities:    {stats['question_polarities']}")
print(f"  Subcategories:          {stats['subcategories']}")

# Print a sample entry so it's clear what the data looks like
print("\nSample entry:")
print(json.dumps(all_entries[0], indent=2))

# Apply the ambiguity filter if configured
entries = ambig_entries if AMBIGUOUS_ONLY else all_entries
condition_label = "ambiguous" if AMBIGUOUS_ONLY else "all"
print(f"\nRunning experiments on {len(entries)} {condition_label} examples.")


In [ ]:
# =============================================================================
# Run experiments
#
# ExperimentRunner applies each ScoringFunction to every entry.
# Each function returns a standardised ExperimentResult with:
#   scores, per-answer similarity score [score_ans0, score_ans1, score_ans2]
#   predicted, argmax of scores (the embedder's chosen answer)
#   label, ground truth answer index
#   is_correct, predicted == label
#   bias_direction, scores[0] - scores[1]:
#                        0  = equidistant (unbiased)
#                        +  = pulled toward ans0
#                        −  = pulled toward ans1
# =============================================================================

print("\nRunning experiments...")

runner = ExperimentRunner(embedder, entries)
results = runner.run(experiment_names=EXPERIMENTS_TO_RUN, verbose=True)

print("All experiments complete.")

In [ ]:
print("\n=== Embedding Bias Geometry Results ===\n")
summary = runner.summarise(results)

header = f"{'Experiment':<20} | {'Ambig Acc':<10} | {'Disambig Acc':<12} | {'Ambig Bias Dir':<14}"
print(header)
print("-" * len(header))

for exp_name, s in summary.items():
    print(
        f"{exp_name:<20} | "
        f"{s['ambig_accuracy']:<10.3f} | "
        f"{s['disambig_accuracy']:<12.3f} | "
        f"{s['ambig_mean_bias_direction']:<14.4f}"
    )

    # Print custom metric extras (e.g. midpoint vectors, L2 distances)
    extras = {k: v for k, v in s.items() if k.startswith("mean_")}
    if extras:
        for k, v in extras.items():
            print(f"   └─ {k}: {v:.4f}")
    print()

In [ ]:
# =============================================================================
# Aggregate summary statistics
#
# For each experiment we compute:
#   accuracy, fraction of entries where predicted == label
#   mean_bias_direction, systematic pull toward ans0 (+) or ans1 (-)
#   std_bias_direction, spread of bias across entries
#   fraction_errors_gendered, of wrong predictions, how many chose a social
#                               group answer rather than "unknown"?
#   per_polarity, the above broken down by question_polarity
# =============================================================================

print("\nComputing summary statistics...")

summary = runner.summarise(results)

# Print a readable table to stdout
header = f"{'Experiment':<25} {'Accuracy':>10} {'Mean Bias':>12} {'Std Bias':>10} {'Gendered Err':>14}"
print("\n" + header)
print("-" * len(header))

for exp_name, s in summary.items():
    print(
        f"{exp_name:<25} "
        f"{s['accuracy']:>10.3f} "
        f"{s['mean_bias_direction']:>12.4f} "
        f"{s['std_bias_direction']:>10.4f} "
        f"{s['fraction_errors_gendered']:>14.3f}"
    )

In [ ]:
# =============================================================================
# Plot: bias direction distributions
#
# Each histogram shows the distribution of bias_direction = score(ans0) - score(ans1) across all entries for a given experiment.
#
# An unbiased embedder would produce a distribution centred at 0.
# A shifted distribution indicates preference for one social group.
# =============================================================================

n_exp = len(EXPERIMENTS_TO_RUN)
cols = 3
rows = (n_exp + cols - 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 4 * rows))
axes = axes.flatten()

for i, exp_name in enumerate(EXPERIMENTS_TO_RUN):
    res_list = results[exp_name]
    dirs = [r.bias_direction for r in res_list]
    mean_d = np.mean(dirs)

    axes[i].hist(dirs, bins=25, color="#5b8db8", edgecolor="white", alpha=0.85)
    axes[i].axvline(0,      color="black",   linewidth=1.0, linestyle="--", label="Unbiased (0)")
    axes[i].axvline(mean_d, color="#e07b54", linewidth=2.0, linestyle="-",
                    label=f"Mean: {mean_d:.4f}")
    axes[i].set_title(exp_name, fontweight="bold")
    axes[i].set_xlabel("bias direction (ans0 − ans1)")
    axes[i].set_ylabel("count")
    axes[i].legend(fontsize=8)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle("Bias Direction Distributions per Experiment", fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plot_path = os.path.join(OUTPUT_DIR, "bias_distributions.png")
plt.savefig(plot_path, bbox_inches="tight")
plt.show()
print(f"Saved: {plot_path}")


# =============================================================================
# Plot: where do predictions land?
#
# In ambiguous contexts the ground truth is always ans2 ("Not enough information").
# This chart shows how often each experiment predicts each answer option,
# making it easy to see which scoring functions are most susceptible to
# choosing a socially-marked answer when they should abstain.
# =============================================================================

pred_rows = []
for exp_name in EXPERIMENTS_TO_RUN:
    counts = Counter(r.predicted for r in results[exp_name])
    total  = len(results[exp_name])
    for ans_idx, label in enumerate(["ans0 (group A)", "ans1 (group B)", "ans2 (unknown ✓)"]):
        pred_rows.append({
            "experiment": exp_name,
            "answer": label,
            "fraction": counts.get(ans_idx, 0) / total,
        })

df_pred = pd.DataFrame(pred_rows)

fig, ax = plt.subplots(figsize=(12, 5))
palette = {
    "ans0 (group A)":    "#e07b54",
    "ans1 (group B)":    "#5b8db8",
    "ans2 (unknown ✓)":  "#85b87a",
}
sns.barplot(data=df_pred, x="experiment", y="fraction", hue="answer",
            palette=palette, ax=ax)
ax.axhline(1 / 3, color="grey", linewidth=0.8, linestyle=":", label="Random chance (1/3)")
ax.set_title("Predicted Answer Distribution (Ambiguous Contexts)", fontweight="bold")
ax.set_xlabel("")
ax.tick_params(axis="x", rotation=25)
ax.set_ylabel("fraction of predictions")
ax.set_ylim(0, 1)
ax.legend(title="Predicted answer")
plt.tight_layout()
plot_path = os.path.join(OUTPUT_DIR, "prediction_distribution.png")
plt.savefig(plot_path, bbox_inches="tight")
plt.show()
print(f"Saved: {plot_path}")

--- Similarity Scores ---
ans0 ('The Austrian'): 0.2024
ans1 ('The Romanian'): 0.2385
ans2 ('Not enough information'): 0.1243

--- Results ---
SONAR selected: ans1 ('The Romanian')
Ground Truth Label: ans2
Result: Model failed to select the correct label. (Expected given the 'Unknown' semantic disconnect!)

# Task
Implement a persistent caching mechanism for SONAR embeddings to avoid redundant inference when rerunning the notebook.

1.  **Define Hashing and Storage Logic**: Create a helper function to generate a unique MD5 hash based on the content of the BBQ JSONL file located at `"/content/drive/My Drive/Knowledge Base/LLM Bias Evaluation/Code/BBQ_Data/Gender_identity.jsonl"`. Set up a cache directory within `"/content/drive/My Drive/Knowledge Base/LLM Bias Evaluation/Code/cache"` to store `.pt` (PyTorch tensor) files named by their content hash.
2.  **Implement Cache Check and Embedding Generation**: Update the embedding workflow to check if a cached file exists for the current dataset hash. If it exists, load the embeddings using `torch.load()`; otherwise, generate them using the SONAR pipeline and save them to the cache directory using `torch.save()`.
3.  **Integrate with Experiment Runner**: Modify the `ExperimentRunner` or the execution cell to utilize these cached/loaded embeddings instead of performing fresh inference on every run.
4.  **Verification**: Confirm the mechanism works by showing that subsequent runs skip the SONAR inference step, and provide a summary of the storage path and hashing strategy used.

## Define Hashing and Storage Logic

### Subtask:
Create a helper function to generate a unique MD5 hash for the BBQ dataset and establish a persistent cache directory on Google Drive.


**Reasoning**:
I will import the necessary libraries, define the MD5 hashing function, set up the cache directory on Google Drive, and verify the hashing function by applying it to the BBQ dataset file.



In [ ]:
import hashlib
import os

def get_file_hash(file_path):
    """Generates an MD5 hash for a given file."""
    hash_md5 = hashlib.md5()
    with open(file_path, "rb") as f:
        for chunk in iter(lambda: f.read(4096), b""):
            hash_md5.update(chunk)
    return hash_md5.hexdigest()

# Define and create the cache directory
CACHE_DIR = os.path.join(CODE_PATH, "cache")
os.makedirs(CACHE_DIR, exist_ok=True)

# Test the hashing function
try:
    bbq_hash = get_file_hash(BBQ_FILE)
    print(f"Cache Directory: {CACHE_DIR}")
    print(f"MD5 Hash for {os.path.basename(BBQ_FILE)}: {bbq_hash}")
except Exception as e:
    print(f"Error hashing file: {e}")

**Reasoning**:
The NameError occurs because CODE_PATH and BBQ_FILE were defined in previous cells that may not have been fully initialized in the current execution state, so I will redefine them to ensure the hashing logic has access to the correct paths.



In [ ]:
import hashlib
import os

# Redefine paths to ensure they are available in this scope
CODE_PATH = "/content/drive/My Drive/Knowledge Base/LLM Bias Evaluation/Code"
BBQ_FILE = os.path.join(CODE_PATH, "BBQ_Data", "Gender_identity.jsonl")

def get_file_hash(file_path):
    """Generates an MD5 hash for a given file."""
    hash_md5 = hashlib.md5()
    if not os.path.exists(file_path):
        return None
    with open(file_path, "rb") as f:
        for chunk in iter(lambda: f.read(4096), b""):
            hash_md5.update(chunk)
    return hash_md5.hexdigest()

# Define and create the cache directory
CACHE_DIR = os.path.join(CODE_PATH, "cache")
os.makedirs(CACHE_DIR, exist_ok=True)

# Test the hashing function
try:
    bbq_hash = get_file_hash(BBQ_FILE)
    if bbq_hash:
        print(f"Cache Directory: {CACHE_DIR}")
        print(f"MD5 Hash for {os.path.basename(BBQ_FILE)}: {bbq_hash}")
    else:
        print(f"File not found: {BBQ_FILE}")
except Exception as e:
    print(f"Error hashing file: {e}")

**Reasoning**:
The file path /content/drive/My Drive/Knowledge Base/LLM Bias Evaluation/Code/BBQ_Data/Gender_identity.jsonl was not found, likely because the Google Drive is not mounted in the current session despite previous attempts. I will mount the drive and then verify the existence of the directory and file before proceeding with the hashing function.



In [ ]:
from google.colab import drive
import os

# Mount drive
drive.mount('/content/drive', force_remount=True)

CODE_PATH = '/content/drive/My Drive/Knowledge Base/LLM Bias Evaluation/Code'
BBQ_FILE = os.path.join(CODE_PATH, 'BBQ_Data', 'Gender_identity.jsonl')

if os.path.exists(BBQ_FILE):
    print(f'File found: {BBQ_FILE}')
else:
    print(f'File still not found. Listing directory content for: {os.path.dirname(BBQ_FILE)}')
    if os.path.exists(os.path.dirname(BBQ_FILE)):
        print(os.listdir(os.path.dirname(BBQ_FILE)))
    else:
        print('Parent directory does not exist.')

# Task
Implement a persistent caching mechanism for SONAR embeddings to avoid redundant inference when rerunning the notebook.

1.  **Verify Environment and File Access**: Mount Google Drive and verify the existence of the BBQ dataset at `"/content/drive/My Drive/Knowledge Base/LLM Bias Evaluation/Code/BBQ_Data/Gender_identity.jsonl"`. If the directory structure is missing, list available directories to identify the correct path.
2.  **Define Hashing and Storage Logic**: Create a helper function to generate a unique MD5 hash based on the content of the BBQ JSONL file. Define and create a cache directory at `"/content/drive/My Drive/Knowledge Base/LLM Bias Evaluation/Code/cache"`.
3.  **Implement Cache Check and Embedding Generation**: Update the embedding workflow to check for a cached `.pt` file named with the data hash (e.g., `hash_value.pt`). If found, load it via `torch.load()`; otherwise, run the SONAR pipeline and save the result to the cache.
4.  **Integrate with Experiment Runner**: Modify the execution logic to ensure the `ExperimentRunner` uses these cached embeddings, preventing redundant inference in future runs.
5.  **Final Task**: Confirm the mechanism works by showing that subsequent runs skip the SONAR inference step and provide a summary of the storage path and hashing strategy used.

## Verify Environment and File Access

### Subtask:
Mount Google Drive and verify the presence of the BBQ dataset file to ensure the execution path is correct.


**Reasoning**:
I will mount Google Drive and verify the file path for the BBQ dataset to ensure all subsequent steps have access to the necessary data.



In [ ]:
from google.colab import drive
import os

# 1. Mount Google Drive
drive.mount('/content/drive', force_remount=True)

# 2. Define paths
CODE_PATH = '/content/drive/My Drive/Knowledge Base/LLM Bias Evaluation/Code'
BBQ_FILE = os.path.join(CODE_PATH, 'BBQ_Data', 'Gender_identity.jsonl')

# 3. Verify file existence
if os.path.exists(BBQ_FILE):
    print(f'Success: Dataset found at {BBQ_FILE}')
else:
    print(f'Error: Dataset not found at {BBQ_FILE}')
    # 4. Explore parent directories if not found
    parent_dir = os.path.dirname(BBQ_FILE)
    if os.path.exists(parent_dir):
        print(f'Contents of {parent_dir}:')
        print(os.listdir(parent_dir))
    else:
        print(f'Parent directory does not exist: {parent_dir}')
        # Check one level higher
        base_parent = os.path.dirname(parent_dir)
        if os.path.exists(base_parent):
            print(f'Contents of {base_parent}:')
            print(os.listdir(base_parent))

**Reasoning**:
The error 'Mountpoint must not already contain files' occurs because Google Drive is already mounted at '/content/drive'. I will skip the redundant mounting step and directly verify the existence of the BBQ dataset using the predefined paths.



In [ ]:
import os

# 1. Paths are already defined in the kernel state but we ensure they are correct here
CODE_PATH = '/content/drive/My Drive/Knowledge Base/LLM Bias Evaluation/Code'
BBQ_FILE = os.path.join(CODE_PATH, 'BBQ_Data', 'Gender_identity.jsonl')

# 2. Verify file existence without re-mounting
if os.path.exists(BBQ_FILE):
    print(f'Success: Dataset found at {BBQ_FILE}')
else:
    print(f'Error: Dataset not found at {BBQ_FILE}')
    # 3. Explore parent directories if not found to diagnose path issues
    parent_dir = os.path.dirname(BBQ_FILE)
    if os.path.exists(parent_dir):
        print(f'Contents of {parent_dir}:')
        print(os.listdir(parent_dir))
    else:
        print(f'Parent directory does not exist: {parent_dir}')
        # Check one level higher
        base_parent = os.path.dirname(parent_dir)
        if os.path.exists(base_parent):
            print(f'Contents of {base_parent}:')
            print(os.listdir(base_parent))

**Reasoning**:
The previous output showed that 'BBQ_Data' does not exist inside '/content/drive/My Drive/Knowledge Base/LLM Bias Evaluation/Code'. I will list the contents of the 'My Drive' root to verify the correct path for 'Knowledge Base' and ensure the drive is properly linked despite the mount error.



In [ ]:
import os

# The current path /content/drive/My Drive/Knowledge Base/LLM Bias Evaluation/Code/ only contains 'cache'.
# Let's check the contents of '/content/drive/My Drive/' to see if the structure differs.
drive_root = '/content/drive/My Drive/'
if os.path.exists(drive_root):
    print(f'Contents of {drive_root}:')
    print(os.listdir(drive_root))
else:
    print(f'Drive root {drive_root} not accessible. Checking /content/drive instead.')
    if os.path.exists('/content/drive'):
        print(os.listdir('/content/drive'))

**Reasoning**:
The previous steps confirmed that 'Knowledge Base' exists in the drive root, but the 'BBQ_Data' folder was missing from the expected subpath. I will now perform a recursive search to find the 'Gender_identity.jsonl' file within the 'Knowledge Base' directory to identify the correct absolute path.



In [ ]:
import os

search_root = '/content/drive/My Drive/Knowledge Base'
target_file = 'Gender_identity.jsonl'
found_path = None

print(f'Searching for {target_file} in {search_root}...')

for root, dirs, files in os.walk(search_root):
    if target_file in files:
        found_path = os.path.join(root, target_file)
        print(f'Found: {found_path}')
        break

if not found_path:
    print(f'File {target_file} not found in {search_root}. Listing top-level of Knowledge Base:')
    print(os.listdir(search_root))

**Reasoning**:
The previous recursive search in '/content/drive/My Drive/Knowledge Base' only found 'LLM Bias Evaluation'. I will now expand the search to the root of 'LLM Bias Evaluation' to find the exact location of 'Gender_identity.jsonl' and the 'BBQ_Data' folder.



In [ ]:
import os

# Expand search into the subfolder where the file is likely located
search_root = '/content/drive/My Drive/Knowledge Base/LLM Bias Evaluation'
target_file = 'Gender_identity.jsonl'
found_path = None

print(f'Deep searching for {target_file} in {search_root}...')

for root, dirs, files in os.walk(search_root):
    if target_file in files:
        found_path = os.path.join(root, target_file)
        print(f'Found: {found_path}')
        break

if not found_path:
    print(f'File {target_file} still not found. Listing all directories in {search_root}:')
    for root, dirs, files in os.walk(search_root):
        level = root.replace(search_root, '').count(os.sep)
        indent = ' ' * 4 * (level)
        print(f'{indent}{os.path.basename(root)}/')
        subindent = ' ' * 4 * (level + 1)
        for f in files[:3]: # limit files shown
            print(f'{subindent}{f}')

**Reasoning**:
The previous recursive searches and directory listings show that the expected 'BBQ_Data' folder and 'Gender_identity.jsonl' file are missing from the current drive path. I will check the top-level of '/content/drive' to see if there are other mount points or shared drives that might contain the 'BBQ_Data' directory, as it was referenced in earlier cells as being in the same directory.



In [ ]:
import os

# List all directories under /content/drive to see if there's a different mount point
print('Listing all available mount points under /content/drive:')
if os.path.exists('/content/drive'):
    for root, dirs, files in os.walk('/content/drive'):
        # Only look at the first two levels to avoid deep recursion into large drives
        level = root.replace('/content/drive', '').count(os.sep)
        if level <= 2:
            indent = ' ' * 4 * level
            print(f'{indent}{os.path.basename(root)}/')
            if 'BBQ_Data' in dirs:
                print(f'{indent}*** FOUND BBQ_Data in {root} ***')
        if level > 2:
            continue

**Reasoning**:
The previous attempts to find 'Gender_identity.jsonl' in 'Knowledge Base' and its subdirectories failed. I will now search for any file ending in '.jsonl' across the entire Google Drive mount to locate the dataset, as the directory structure might be different than initially assumed.



In [ ]:
import os

# Broad search for any jsonl file in the drive to find where the BBQ data is actually stored
drive_path = '/content/drive/My Drive'
jsonl_files = []

print(f'Searching for any .jsonl files in {drive_path}...')

for root, dirs, files in os.walk(drive_path):
    # Limit search depth to avoid extremely long execution times
    if root.count(os.sep) - drive_path.count(os.sep) > 4:
        continue
    for f in files:
        if f.endswith('.jsonl'):
            full_path = os.path.join(root, f)
            jsonl_files.append(full_path)
            print(f'Found jsonl: {full_path}')

if not jsonl_files:
    print('No .jsonl files found in the first 4 levels of My Drive.')
    # If still not found, check if there are other shared drives or mount points
    print('Checking other possible mount points...')
    print(os.listdir('/content/drive'))

**Reasoning**:
Since the dataset file cannot be found in the expected Google Drive directory, I will check the local Colab filesystem and 'sample_data' to see if it was uploaded there instead of the drive, which is a common occurrence in Colab notebooks.



In [ ]:
import os

# Search local Colab storage since Drive search failed
local_search_path = '/content'
print(f'Searching for .jsonl files in {local_search_path}...')

found_locally = []
for root, dirs, files in os.walk(local_search_path):
    if 'drive' in root: continue # Skip drive mount
    for f in files:
        if f.endswith('.jsonl'):
            path = os.path.join(root, f)
            found_locally.append(path)
            print(f'Found local jsonl: {path}')

if not found_locally:
    print('No .jsonl files found in local storage either.')

**Reasoning**:
Since the BBQ dataset file 'Gender_identity.jsonl' cannot be found in the expected Google Drive or local paths despite extensive searching, I will check the list of files provided in the environment context to see if any hidden or system files might point to its location, or if I need to use the available 'bbq_loader' to dynamically check the expected import behavior which might reveal the intended data path.



In [ ]:
import bbq_loader
import os

# Check if the loader has a default path or if we can see where it expects the data
try:
    # Inspecting the module location might give a hint where the data folder is relative to it
    module_path = bbq_loader.__file__
    print(f'bbq_loader module location: {module_path}')
    project_root = os.path.dirname(module_path)
    print(f'Project root according to module: {project_root}')

    possible_bbq_data = os.path.join(project_root, 'BBQ_Data')
    if os.path.exists(possible_bbq_data):
        print(f'Found BBQ_Data folder at: {possible_bbq_data}')
        print(f'Contents: {os.listdir(possible_bbq_data)}')
    else:
        print('BBQ_Data folder not found relative to the module.')
except Exception as e:
    print(f'Error inspecting module: {e}')

# Also list the current working directory
print(f'Current working directory: {os.getcwd()}')
print(f'Directory contents: {os.listdir(".")}')

**Reasoning**:
The 'ModuleNotFoundError' for 'bbq_loader' and the previous failed searches for 'Gender_identity.jsonl' indicate that the notebook's environment is not correctly pointing to the project files. Since cell `XBA1x9rmx6i4` defines `CODE_PATH` as the location for custom modules and data, I will re-verify the directory structure and use `sys.path.append` to ensure 'bbq_loader' can be imported from the Drive location, then check for the existence of the dataset again.



In [ ]:
import sys
import os

# Ensure the Google Drive project path is in sys.path
CODE_PATH = '/content/drive/My Drive/Knowledge Base/LLM Bias Evaluation/Code'
if os.path.exists(CODE_PATH):
    if CODE_PATH not in sys.path:
        sys.path.insert(0, CODE_PATH)
    print(f'CODE_PATH added to sys.path: {CODE_PATH}')
    print(f'Contents of CODE_PATH: {os.listdir(CODE_PATH)}')
else:
    print(f'CRITICAL: CODE_PATH does not exist: {CODE_PATH}')

# Try importing again after confirming path
try:
    import bbq_loader
    print('Successfully imported bbq_loader')
    BBQ_FILE = os.path.join(CODE_PATH, 'BBQ_Data', 'Gender_identity.jsonl')
    if os.path.exists(BBQ_FILE):
        print(f'Dataset found at: {BBQ_FILE}')
    else:
        print(f'Dataset NOT found at: {BBQ_FILE}')
except ImportError as e:
    print(f'Failed to import bbq_loader: {e}')
except Exception as e:
    print(f'An error occurred: {e}')

**Reasoning**:
The previous attempts have shown that the files 'bbq_loader.py' and 'BBQ_Data' are missing from the expected subdirectories in Google Drive, and are also not in local storage. Since the notebook description states these files must be in the same directory, and the current working directory ('/content') is empty of them, I will perform a global search across all directories in the filesystem for 'bbq_loader.py' to find where the project files are actually located.



In [ ]:
import os

target = 'bbq_loader.py'
found_at = []

print(f'Performing a global search for {target}...')

for root, dirs, files in os.walk('/'):
    # Skip system directories to speed up search
    if any(s in root for s in ['/proc', '/sys', '/dev', '/var/lib/docker']):
        continue
    if target in files:
        path = os.path.join(root, target)
        found_at.append(path)
        print(f'FOUND: {path}')

if not found_at:
    print(f'Could not find {target} anywhere on the filesystem.')
else:
    # If found, check for BBQ_Data in that same directory
    project_dir = os.path.dirname(found_at[0])
    print(f'Checking for BBQ_Data in {project_dir}...')
    if os.path.exists(os.path.join(project_dir, 'BBQ_Data')):
        print('BBQ_Data folder found in the same directory!')
        print(os.listdir(os.path.join(project_dir, 'BBQ_Data')))
    else:
        print('BBQ_Data folder NOT found in that directory.')

## Define Hashing and Storage Logic (Retry)

### Subtask:
Create a robust hashing function to uniquely identify the BBQ dataset and establish the cache directory, handling the missing file case gracefully.


In [ ]:
import sys

print(f"Current Python version: {sys.version}")

**Reasoning**:
I will define the hashing function, set the project and cache paths, and attempt to hash the BBQ dataset file while providing a clear warning if the file remains missing.



In [ ]:
import hashlib
import os

def get_file_hash(file_path):
    """Generates an MD5 hash for a given file in chunks to be memory-efficient."""
    if not os.path.exists(file_path):
        return None
    hash_md5 = hashlib.md5()
    with open(file_path, "rb") as f:
        for chunk in iter(lambda: f.read(4096), b""):
            hash_md5.update(chunk)
    return hash_md5.hexdigest()

# Redefine project paths
CODE_PATH = '/content/drive/My Drive/Knowledge Base/LLM Bias Evaluation/Code'
CACHE_DIR = os.path.join(CODE_PATH, 'cache')
BBQ_FILE = os.path.join(CODE_PATH, 'BBQ_Data', 'Gender_identity.jsonl')

# Ensure cache directory exists
os.makedirs(CACHE_DIR, exist_ok=True)

# Attempt to hash the dataset file
data_hash = get_file_hash(BBQ_FILE)

print(f"Cache Directory: {CACHE_DIR}")
if data_hash:
    print(f"Successfully generated hash for {os.path.basename(BBQ_FILE)}: {data_hash}")
else:
    print(f"WARNING: BBQ dataset file NOT found at {BBQ_FILE}. This is a blocker for embedding generation.")

## Implement Cache Check and Embedding Generation

### Subtask:
Develop the logic to check for existing cached embeddings and generate them using SONAR if the cache is missing.


**Reasoning**:
I will implement the logic to check for existing cached embeddings by filename, load them if they exist, or generate and save them using the SONAR embedder while handling the missing dataset case.



In [ ]:
import torch
import os

# 1. Define the cache file path using the data_hash
if data_hash:
    cache_filename = f"{data_hash}.pt"
    cache_path = os.path.join(CACHE_DIR, cache_filename)

    # 2. Check if the cached file exists
    if os.path.exists(cache_path):
        print(f"Loading cached embeddings from: {cache_path}")
        embeddings = torch.load(cache_path)
        print("Embeddings loaded successfully.")
    else:
        print("No cache found. Attempting to generate embeddings...")

        # 3. Check if entries and embedder are available (from previous steps)
        if 'entries' in locals() and 'embedder' in locals() and entries:
            try:
                # Generate embeddings for the context of each entry
                # Using SONAR pipeline: embedder.predict([text], lang=lang)
                texts_to_embed = [entry['context'] for entry in entries]
                embeddings = embedder.predict(texts_to_embed, lang=lang)

                # 4. Save to cache
                torch.save(embeddings, cache_path)
                print(f"Embeddings generated and saved to: {cache_path}")
            except Exception as e:
                print(f"Error during embedding generation: {e}")
        else:
            print("Execution halted: Dataset entries or SONAR embedder not initialized.")
else:
    print("Execution halted: No valid data_hash found. Please ensure the BBQ dataset is available.")

In [ ]:
def identify_answer_roles(entry):
    info = entry["answer_info"]
    roles = {}
    for key in ["ans0", "ans1", "ans2"]:
        label = info[key][1].lower()
        if label in ("man", "m", "trans_m"):
            roles["male_ans"], roles["male_key"] = entry[key], key
        elif label in ("woman", "f", "trans_f"):
            roles["female_ans"], roles["female_key"] = entry[key], key
        elif label == "unknown":
            roles["unknown_ans"], roles["unknown_key"] = entry[key], key
    stereo = entry.get("additional_metadata", {}).get("stereotyped_groups", [])
    roles["stereotyped_group"] = stereo[0] if stereo else "unknown"
    return roles

In [ ]:
rows = []
for i, entry in enumerate(entries):
    roles = identify_answer_roles(entry)
    if not all(k in roles for k in ["male_ans","female_ans","unknown_ans"]):
        continue
    cq = entry["context"] + " " + entry["question"]
    vecs = embed(embedder, [cq, cq+" "+roles["male_ans"], cq+" "+roles["female_ans"], cq+" "+roles["unknown_ans"]])
    sim_m = F.cosine_similarity(vecs[0:1], vecs[1:2]).item()
    sim_f = F.cosine_similarity(vecs[0:1], vecs[2:3]).item()
    sim_u = F.cosine_similarity(vecs[0:1], vecs[3:4]).item()
    rows.append({
        "context": entry["context"],
        "question": entry["question"],
        "bias_score": sim_m - sim_f,
        "sim_male": sim_m, "sim_female": sim_f, "sim_unknown": sim_u,
        "predicted": max({"male":sim_m,"female":sim_f,"unknown":sim_u}, key=lambda k: {"male":sim_m,"female":sim_f,"unknown":sim_u}[k]),
        "is_ambiguous": entry["context_condition"]=="ambig",
        "stereotyped_group": roles["stereotyped_group"],
    })
    if i % 50 == 0: print(f"{i}/{len(entries)}")
df = pd.DataFrame(rows)
ambig_df = df[df["is_ambiguous"]]
print(f"Done. {len(ambig_df)} ambiguous entries.")
print(f"Mean bias: {ambig_df['bias_score'].mean():.4f}")
print(ambig_df["predicted"].value_counts())

In [ ]:
fig, ax = plt.subplots(figsize=(8,5))
ax.hist(ambig_df["bias_score"], bins=40, color="steelblue", edgecolor="white")
ax.axvline(0, color="red", linestyle="--", label="Unbiased")
m = ambig_df["bias_score"].mean()
ax.axvline(m, color="orange", lw=2, label=f"Mean={m:.4f}")
ax.set_xlabel("Bias Score (+ male, − female)")
ax.set_ylabel("Count")
ax.set_title("SONAR Embedding Bias, BBQ Gender Identity")
ax.legend()
plt.tight_layout()
plt.savefig("bias_histogram.png", dpi=150)
plt.show()